# Outline
* Required Installation
* Mathematical Background
  * Additive Delay Model
  * Reliability
  * Uniqueness
  * Uniformity
  * Modeling Attack: Logistic Regression
* XOR Arbiter PUF (XOR-APUF)
  * Additive Delay Model
  * PyPUF Implementation
  * Security Analysis

## Required Installation

Run the cell below once to install the required packages:

- `pypuf==3.2.1` — PUF simulations (`XORArbiterPUF`), quality metrics and the `LRAttack2021` modeling attack
- `tensorflow` — the training backend used by `LRAttack2021`
- `numpy<2.0` — pypuf 3.2.1 is not compatible with NumPy 2.x
- `matplotlib` — plotting

In [ ]:
%pip install pypuf==3.2.1
%pip install tensorflow "numpy<2.0" matplotlib

In [1]:
import numpy as np

from pypuf.simulation import XORArbiterPUF
from pypuf.io import random_inputs
from pypuf.metrics import reliability, uniqueness, similarity_data

np.set_printoptions(precision=4, suppress=True)

## 1. Mathematical Background
### 1.1. Additive Delay Model
The Additive Delay Model is the general framework used across the whole family of **delay-based PUFs**.

**Core idea:** the physical quantity that decides the output (delay difference, phase difference, etc.) is a **linear combination** of a challenge-derived feature vector and a chip-specific weight vector:

$$\Delta = \mathbf{w}^T \boldsymbol{\phi}(\mathbf{c}) + w_0, \qquad r_{\pm} = \text{sign}(\Delta) \in \{-1,+1\}$$

- $\mathbf{w}$: fixed but unknown, chip-specific weights (encodes physical delay/phase variations)
- $\boldsymbol{\phi}(\mathbf{c})$: an **architecture-specific** transform of the challenge
- $w_0$: bias term

**Response encoding (used throughout both labs, identical to pypuf):** responses are handled in the bipolar form $r_{\pm}\in\{-1,+1\}$, and the corresponding bit is

$$r = \frac{1-r_{\pm}}{2}, \qquad \text{i.e.}\quad +1 \leftrightarrow 0,\quad -1 \leftrightarrow 1, \qquad r = 1 \iff \Delta < 0$$

In code this is `bits = (1 - responses) // 2`, and the reverse is `responses = 1 - 2 * bits`.

**Why this abstraction matters:**
1. **ML attack analysis generalizes** — any PUF fitting $\Delta = \mathbf{w}^T\phi+w_0$ is vulnerable to the same attack class (logistic regression, SVM), regardless of architecture.
2. **Security comparison reduces to one question:** how non-linear can $\phi(\mathbf{c})$ (or $\Delta$'s dependence on it) be made? XOR-PUF and Feed-Forward PUF are attempts to break this linearity without abandoning the additive skeleton.

**In short:** the Additive Delay Model abstracts any delay-based PUF as $\Delta = \mathbf{w}^T\phi(\mathbf{c})+w_0,\ r_{\pm}=\text{sign}(\Delta)$; APUF is the special case where $\phi$ is the simple parity vector, while other architectures either redefine $\phi$ or combine multiple additive components non-linearly (e.g. via XOR).

### 1.2. Reliability
**Definition:** Measures how stable a single PUF instance's responses are when the same challenge is queried multiple times under simulated noise.

**Theoretical formula**:

$$\text{Reliability} = \Pr_{\text{eval}}\left[\text{eval}(x) = \text{eval}(x)\right]$$

That is, the probability that the same challenge $x$, evaluated twice under the noisy evaluation process, yields the same result.

In [2]:
# Reliability test.
#
# IMPORTANT: `instance` must have been created with `noisiness > 0`
# (e.g. ArbiterPUF(..., noisiness=0.05)). A noiseless instance
# (noisiness=0, the default) will always score a perfect 1.0, which is
# not informative.
#
# Returns the overall reliability score in [0, 1]. Ideal: 1.0.
def test_reliability(instance, seed: int = 10, N: int = 1000, r: int = 17,
                      verbose: bool = True) -> float:
    rel_per_challenge = reliability(instance, seed=seed, N=N, r=r)
    overall = float(np.average(rel_per_challenge))

    if verbose:
        print(f"[Reliability] Overall score : {overall:.4f}  (ideal: 1.0000)")
        print(f"[Reliability] Min observed  : {rel_per_challenge.min():.4f}")
        print(f"[Reliability] Max observed  : {rel_per_challenge.max():.4f}")

    return overall

### 1.3. Uniqueness

**Definition:** Measures how distinguishable a set of PUF instances (representing different physical chips of the same design) are from one another, based on their responses to the same set of random challenges.

**First, pairwise similarity between two chips is defined:**

$$\text{sim}(f,g) = \Pr_x\left[f(x) = g(x)\right] \approx \frac{1}{N}\sum_{i=1}^{N} \mathbb{1}[f(c_i) = g(c_i)]$$

**Uniqueness formula**, averaged over all chip pairs:

$$\text{Uniqueness} = 1 - 2 \cdot \mathbb{E}_{f \neq g}\left[\left|\tfrac{1}{2} - \text{sim}(f,g)\right|\right]$$

**Intuition:**
- Ideally, two different chips behave like independent coin flips on a random challenge, i.e. $\text{sim}(f,g) = 0.5$ → $|\tfrac{1}{2}-0.5|=0$ → Uniqueness $=1$.
- If two chips always agree ($\text{sim}=1$) or always disagree ($\text{sim}=0$), in both cases $|\tfrac{1}{2}-\text{sim}| = 0.5$ → Uniqueness $= 1 - 2(0.5) = 0$ (worst case: one chip's responses are fully determined by the other's — identical in the first case, exactly inverted in the second).

For $l$ chips, the average is taken over all $\binom{l}{2}$ pairs.

In [3]:
# Uniqueness test
#
# `instances` must contain at least 2 PUF instances (e.g. created with
# different seeds, each representing a different "chip").
#
# Returns the overall uniqueness score in [0, 1]. Ideal: 1.0.
def test_uniqueness(instances: list, seed: int = 20, N: int = 2000,
                     verbose: bool = True) -> float:
    if len(instances) < 2:
        raise ValueError("Uniqueness test requires at least 2 PUF instances.")

    score = float(uniqueness(instances, seed=seed, N=N)[0])

    if verbose:
        print(f"[Uniqueness]  Score ({len(instances)} chips): {score:.4f}  "
              f"(ideal: 1.0000)")

    return score

### 1.4. Uniformity

**Definition:** Measures the balance of 0s and 1s in a single PUF instance's responses over a large random challenge set. Bias is measured by how far this value deviates from $0.5$.

$$\text{Uniformity} = \frac{1}{N}\sum_{i=1}^{N} r_i \quad, \quad r_i \in \{0,1\}$$

where $r_i$ is the response **bit** (encoding from Section 1.1). Ideal value: $0.5$.



In [4]:
# Uniformity test.
#
# pypuf has no function for this, so it is computed directly.
# pypuf encodes responses as -1/+1 with bit = (1 - r) // 2,
# i.e. +1 -> 0 and -1 -> 1 (see Section 1.1).
#
# Returns the fraction of response bits equal to 1. Ideal: 0.5.

def test_uniformity(instance, n: int, N: int = 5000, seed: int = 30,
                     verbose: bool = True) -> float:
    challenges = random_inputs(n=n, N=N, seed=seed)
    responses = instance.eval(challenges)  # values are -1 / +1

    responses_binary = ((1 - responses) // 2).astype(int)  # -1/+1 -> 1/0
    score = float(responses_binary.mean())

    if verbose:
        deviation = abs(score - 0.5)
        print(f"[Uniformity]  Score: {score:.4f}  (ideal: 0.5000)  "
              f"-> {score*100:.2f}% ones / {(1-score)*100:.2f}% zeros, "
              f"deviation: {deviation:.4f}")

    return score


### 1.5. Modeling Attack (Logistic Regression Attack)

**Source of the weakness — additive delay model:**

$$\Delta = \mathbf{w}^T \boldsymbol{\phi}(\mathbf{c}) + w_0$$

Response bit (encoding from Section 1.1):
$$r = \frac{1-\text{sign}(\Delta)}{2} = \begin{cases}1 & \Delta<0\\0 & \Delta\geq0\end{cases}$$

**Attacker's goal:** without knowing the true $\mathbf{w}$, estimate $\hat{\mathbf{w}}$ from a set of $(\mathbf{c}_i, r_i)$ pairs.

**The logistic regression model** turns the estimated delay difference $\hat\Delta = \hat{\mathbf{w}}^T \boldsymbol{\phi} + \hat{w}_0$ into a probability with a sigmoid. Since $r=1 \iff \Delta<0$, the sigmoid is applied to $-\hat\Delta$:

$$P(r=1 \mid \boldsymbol{\phi}) = \sigma\big(-(\hat{\mathbf{w}}^T \boldsymbol{\phi} + \hat{w}_0)\big) = \frac{1}{1+e^{\,\hat{\mathbf{w}}^T \boldsymbol{\phi}+\hat{w}_0}}$$

**Training minimizes the cross-entropy (log-loss):**

$$\mathcal{L}(\hat{\mathbf{w}}) = -\frac{1}{N}\sum_{i=1}^{N}\Big[r_i \log P(\boldsymbol{\phi}_i) + (1-r_i)\log\big(1-P(\boldsymbol{\phi}_i)\big)\Big]$$

**Why it works so well:** since $\Delta$ is a **linear** function of $\boldsymbol{\phi}(\mathbf{c})$, logistic regression's decision boundary ($\hat{\mathbf{w}}^T\boldsymbol{\phi}+\hat{w}_0 = 0$) belongs to the **exact same hypothesis class** as the PUF's true decision boundary ($\mathbf{w}^T\boldsymbol{\phi}+w_0=0$). This is why gradient descent can converge to $\hat{\mathbf{w}} \approx c\cdot\mathbf{w}$ (for some positive scaling constant $c$) with a finite number of CRPs.

**From one APUF to a $k$-XOR-APUF — what `LRAttack2021` actually trains:** for $k>1$ the PUF's decision boundary is no longer a single hyperplane in $\boldsymbol{\phi}$-space (see Section 2.1), so a single logistic regression cannot represent it. pypuf's `LRAttack2021` (a TensorFlow re-implementation of the logistic regression attack of Rührmair et al., 2010) therefore keeps one linear model per component and combines them the same way the PUF does — as a product in the bipolar domain:

$$\hat r_{\pm}(\mathbf{c}) = \tanh\!\Big(\prod_{l=1}^{k} \tanh\big(\hat{\mathbf{w}}^{(l)T}\boldsymbol{\phi}(\mathbf{c})+\hat w_0^{(l)}\big)\Big)$$

All $k$ weight vectors are trained jointly with the same cross-entropy loss on the bit labels $r=(1-r_\pm)/2$ (the model's output is mapped to $P(r=1) = \tfrac12 - \tfrac12\hat r_{\pm}$). For $k=1$ the model reduces to the logistic regression above ($\tanh$ is a rescaled sigmoid). For $k>1$ the loss is **non-convex**: the required number of CRPs grows quickly with $k$, and a single training run can occasionally get stuck in a poor local optimum.

**Attack success metric** (same formula as the similarity function):
$$\text{Attack Accuracy} = \frac{1}{N_{\text{test}}}\sum_{i=1}^{N_{\text{test}}} \mathbb{1}\big[\hat{r}_i = r_i\big]$$


In [5]:
import matplotlib.pyplot as plt
import tensorflow as tf
from pypuf.simulation.base import LTFArray
from pypuf.io import random_inputs, ChallengeResponseSet
from pypuf.attack import LRAttack2021
import pypuf.attack

# Modeling attack with LRAttack2021 (see Section 1.5).
#
# Collects N challenge-response pairs from `instance` (seed `train_seed`)
# and trains LRAttack2021 on all of them (LRAttack2021 internally holds out
# 1% of them as a validation set). The model is then evaluated on a
# separate, freshly generated test set of int(N * test_ratio) challenges
# (seed `fresh_seed`, independent of the training challenges).
#
# Prints the attack accuracy on the fresh test set; returns nothing.
def run_ml_attack(instance, n: int, k: int, N: int = 50000, test_ratio: float = 0.2,
                   train_seed: int = 2, fresh_seed: int = 5) -> None:
    train_challenges = random_inputs(n=n, N=N, seed=train_seed)
    train_responses = instance.eval(train_challenges)

    test_N = int(N * test_ratio)
    test_challenges = random_inputs(n=n, N=test_N, seed=fresh_seed)
    test_responses = instance.eval(test_challenges)

    crps = ChallengeResponseSet(train_challenges, train_responses)
    attack = LRAttack2021(crps, seed=100, k=k, bs=1000, lr=0.001, epochs=50, stop_validation_accuracy=1.0)
    model = attack.fit()

    predicted = model.eval(test_challenges)
    accuracy = np.mean(
        predicted.reshape(-1) == test_responses.reshape(-1)
    )
    print('Attack accuracy:', accuracy)


I0000 00:00:1790972412.743406   89345 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1790972412.780821   89345 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1790972413.819560   89345 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


## 2. XOR Arbiter PUF (XOR-APUF)

**Definition:** An XOR Arbiter PUF combines $k$ **independent** classical Arbiter PUFs (APUFs), each fed with the **same challenge**, by **XOR-ing** their outputs together.

![image](./Pictures/PUFscheme.png)

### 2.1. Additive Delay Model

First, we derive the **Additive Delay Model** of a single Arbiter PUF.

Let $\Delta_{i-1}$ be the delay difference entering stage $i$. The new difference $\Delta_i$ leaving that stage depends on the challenge bit $c_i$:

- **$c_i=0$ (straight):** signals don't swap → $\Delta_i = \Delta_{i-1} + \delta_i^{0}$
- **$c_i=1$ (crossed):** signals swap, so the previous difference's **sign flips** (the leading signal is now on the other line) → $\Delta_i = -\Delta_{i-1} + \delta_i^{1}$

Both cases combine into a single formula:

$$\Delta_i = (1-2c_i)\,\Delta_{i-1} + \delta_i(c_i), \qquad \delta_i(c_i) = (1-2c_i)\alpha_i + \beta_i$$

where:
- $\alpha_i$: the "straight-through" delay-difference component of that stage (chip-specific, due to manufacturing variation)
- $\beta_i$: a fixed delay component introduced by the crossover itself

Starting from $\Delta_0 = 0$ and unrolling for $i=1,\dots,n$:

$$\Delta_n = \sum_{i=1}^{n} \delta_i(c_i) \cdot \prod_{j=i+1}^{n}(1-2c_j)$$

Each stage's contribution ($\delta_i$) enters the sum multiplied by a coefficient that depends on **how many times all subsequent stages flip the sign**. This "product of subsequent stages" term is exactly the **parity vector**:

$$\phi_i = \prod_{j=i}^{n} (1-2c_j), \qquad i=1,\dots,n, \qquad \phi_{n+1} = 1$$

Substituting $\delta_i(c_i) = (1-2c_i)\alpha_i + \beta_i$: the $\alpha_i$ term carries the extra factor $(1-2c_i)$ and becomes $\alpha_i\phi_i$, while the $\beta_i$ term does not and becomes $\beta_i\phi_{i+1}$:

$$\Delta_n = \sum_{i=1}^{n}\big(\alpha_i\,\phi_i + \beta_i\,\phi_{i+1}\big)$$

Grouping the coefficients of each $\phi_i$ ($\beta_{i-1}$ lands on $\phi_i$, and $\beta_n$ multiplies the constant $\phi_{n+1}=1$):

$$\boxed{\Delta = \mathbf{w}^T \boldsymbol{\phi}(\mathbf{c}) + w_0, \qquad w_1=\alpha_1,\quad w_i=\alpha_i+\beta_{i-1}\ \ (i=2,\dots,n),\quad w_0=\beta_n}$$

$$r_{\pm} = \text{sign}(\Delta), \qquad r = \frac{1-r_{\pm}}{2} = \begin{cases} 1 & \Delta < 0 \\ 0 & \Delta \geq 0 \end{cases}$$

where:
- $\mathbf{w}=(w_1,\dots,w_n)$: a chip-specific weight vector, fixed at manufacturing time (a combination of the stage parameters $\alpha_i, \beta_i$)
- $\boldsymbol{\phi}(\mathbf{c})$: the architecture-specific parity vector computed from the challenge
- $w_0 = \beta_n$: a constant bias term

In pypuf, challenges are already given in $\{-1,+1\}$ form, so $(1-2c_j)$ is simply $c_j$ and $\phi_i=\prod_{j=i}^{n}c_j$ (this is `XORArbiterPUF.transform_atf`).

The following lines describe the XOR-APUF **Additive Delay Model**.

Each component ($l = 1, \dots, k$) follows the standard APUF additive model on its own:

$$\Delta^{(l)} = \mathbf{w}^{(l)T} \boldsymbol{\phi}(\mathbf{c}) + w_0^{(l)}$$

Each component's own response (bipolar and bit form, Section 1.1):
$$r_{\pm}^{(l)} = \text{sign}\left(\Delta^{(l)}\right), \qquad r^{(l)} = \frac{1-r_{\pm}^{(l)}}{2}$$

The final response is their XOR:
$$r = r^{(1)} \oplus r^{(2)} \oplus \dots \oplus r^{(k)}$$

Since XOR becomes a **product** in the bipolar $\{-1,+1\}$ representation ($(-1)^{a \oplus b} = (-1)^a(-1)^b$):

$$r_{\pm} = \prod_{l=1}^{k} r_{\pm}^{(l)} = \prod_{l=1}^{k} \text{sign}\left(\mathbf{w}^{(l)T}\boldsymbol{\phi}(\mathbf{c}) + w_0^{(l)}\right)$$

This holds for any $k$, odd or even, and is exactly what pypuf's `XORArbiterPUF.eval` computes.

A short comparison between the classical APUF and the XOR-APUF:

| | Classical APUF | XOR-APUF |
|---|---|---|
| $\Delta$ | A single linear sum: $\mathbf{w}^T\phi+w_0$ | **Product of the signs** of $k$ separate linear sums |
| Linearity | Fully linear in $\phi(\mathbf{c})$ | Linear within each component, but the combination is non-linear |
| ML resistance | Weak (broken by simple logistic regression) | Increases with $k$ (required CRP count grows exponentially) |
| Reliability | High | Decreases as $k$ grows (noise accumulates through the XOR) |

### 2.2. PyPUF Implementation
Create a 64-bit XOR Arbiter PUF with $k=5$ components and generate a batch of challenge-response pairs (CRPs).

In [6]:
n = 64  # challenge length / number of MUX stages
k = 5
puf = XORArbiterPUF(n=n, k=k,seed=1)
print(f"[+] Simulated a {n}-bit XOR Arbiter PUF with k={k} (seed=1)")

challenges = random_inputs(n=n, N=200000, seed=2)
responses = puf.eval(challenges)
print(f"[+] Generated {len(challenges)} challenge-response pairs")
print(f"    Example challenge (first 10 bits): {challenges[0][:10]}")
print(f"    Corresponding response           : {responses[0]}")

[+] Simulated a 64-bit XOR Arbiter PUF with k=5 (seed=1)
[+] Generated 200000 challenge-response pairs
    Example challenge (first 10 bits): [-1 -1  1  1  1 -1  1  1  1  1]
    Corresponding response           : -1


## 2.3. Security Analysis
### 2.3.1. Reliability 

In [7]:
# Reliability requires a NOISY instance (same seed = same "chip",
# but with simulated environmental noise turned on)
noisy_puf = XORArbiterPUF(n=n, k=k, seed=1, noisiness=0.05)
test_reliability(noisy_puf, seed=10, N=1000, r=17)

[Reliability] Overall score : 0.9068  (ideal: 1.0000)
[Reliability] Min observed  : 0.5017
[Reliability] Max observed  : 1.0000


0.9068235294117646

### 2.3.2. Uniqueness

In [8]:
# Uniqueness requires MULTIPLE chip instances (different seeds)
chip_instances = [XORArbiterPUF(n=n, k=k, seed=s) for s in range(100, 110)]
test_uniqueness(chip_instances, seed=20, N=2000)

[Uniqueness]  Score (10 chips): 0.9814  (ideal: 1.0000)


0.9814444444444445

### 2.3.3. Uniformity

In [9]:
# Uniformity is evaluated on a single chip instance
test_uniformity(puf, n=n, N=5000, seed=30)

[Uniformity]  Score: 0.5016  (ideal: 0.5000)  -> 50.16% ones / 49.84% zeros, deviation: 0.0016


0.5016

### 2.3.4. Modeling Attack: Logistic Regression (`LRAttack2021`)

In [12]:
run_ml_attack(puf, n=n, k=k, N=200000)

Epoch 1/50
198/198 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - accuracy: 0.4998 - loss: 0.6932 - val_accuracy: 0.4940 - val_loss: 0.6933
Epoch 2/50
198/198 ━━━━━━━━━━━━━━━━━━━━ 0s 960us/step - accuracy: 0.5051 - loss: 0.6930 - val_accuracy: 0.4880 - val_loss: 0.6933
Epoch 3/50
198/198 ━━━━━━━━━━━━━━━━━━━━ 0s 950us/step - accuracy: 0.5066 - loss: 0.6928 - val_accuracy: 0.4950 - val_loss: 0.6933
Epoch 4/50
198/198 ━━━━━━━━━━━━━━━━━━━━ 0s 950us/step - accuracy: 0.5096 - loss: 0.6925 - val_accuracy: 0.4860 - val_loss: 0.6934
Epoch 5/50
198/198 ━━━━━━━━━━━━━━━━━━━━ 0s 962us/step - accuracy: 0.5112 - loss: 0.6923 - val_accuracy: 0.4720 - val_loss: 0.6935
Epoch 6/50
198/198 ━━━━━━━━━━━━━━━━━━━━ 0s 981us/step - accuracy: 0.5117 - loss: 0.6921 - val_accuracy: 0.4710 - val_loss: 0.6936
Epoch 7/50
198/198 ━━━━━━━━━━━━━━━━━━━━ 0s 956us/step - accuracy: 0.5146 - loss: 0.6920 - val_accuracy: 0.4730 - val_loss: 0.6937
Epoch 8/50
198/198 ━━━━━━━━━━━━━━━━━━━━ 0s 970us/step - accuracy: 0.5146 - loss: 0.6919 - va

**WARNING:** If you encounter an error like; 


"Traceback (most recent call last):
  File "../QPUF Tutorial/deneme.py", line 27, in <module>
    model = attack.fit()
  File "../.local/lib/python3.10/site-packages/pypuf/attack/lr2021.py", line 145, in fit
    self._history = model.fit(
  File "../.local/lib/python3.10/site-packages/keras/src/utils/traceback_utils.py", line 122, in error_handler
    raise e.with_traceback(filtered_tb) from None
  File "../.local/lib/python3.10/site-packages/pypuf/attack/lr2021.py", line 89, in accuracy
    return tf.keras.metrics.binary_accuracy(.5 - .5 * y_true, .5 - .5 * y_pred)
TypeError: Expected int8, but got 0.5 of type 'float'."

run following cell and train again.

In [11]:
def keras_to_pypuf_fixed(self, keras_model):
    n, k = self.crps.challenges.shape[1], self.k
    weights = np.zeros(shape=(k, n))
    bias = np.zeros(shape=(k,))
    for l in range(k):
        layer_weights = keras_model.layers[l + 1].get_weights()
        weights[l] = layer_weights[0][:, 0]
        bias[l] = layer_weights[1][0]
    return LTFArray(weight_array=weights, bias=bias, transform=XORArbiterPUF.transform_atf)
pypuf.attack.LRAttack2021.keras_to_pypuf = keras_to_pypuf_fixed

@staticmethod
def loss(y_true, y_pred):
    y_true = tf.cast(y_true, tf.float32)
    return tf.keras.losses.binary_crossentropy(.5 - .5 * y_true, .5 - .5 * y_pred)

@staticmethod
def accuracy(y_true, y_pred):
    y_true = tf.cast(y_true, tf.float32)
    return tf.keras.metrics.binary_accuracy(.5 - .5 * y_true, .5 - .5 * y_pred)

pypuf.attack.LRAttack2021.loss = loss
pypuf.attack.LRAttack2021.accuracy = accuracy